# Cubo Gelatinoso
Heloísa Fernandes Cano - T26

## Enunciado

### Objetivo

<b>Objetivo:</b> Escolha um conjunto de dados que siga as considerações abaixo e estude o desempenho de modelos induzidos pelo algoritmo de <i>k</i> vizinhos mais próximos neste conjunto de dados 
considerando diferentes conjuntos de hiperparâmetros (pelo menos 10 conjuntos de hiperparâmetros 
diferentes). A entrega deve ser realizada em um notebook de Jupyter. A sugestão é que esta seja uma 
entrega com texto direto, sem a necessidade de longos parágrafos. É necessário utilizar o `scikit‑learn`
para induzir os modelos de <i>k</i> vizinhos mais próximos. Não se esqueça de justificar a métrica de 
desempenho escolhida.

### Considerações

<b>Considerações:</b> O conjunto de dados deve ter as seguintes características:
- Pelo menos 5 atributos. Destes atributos, pelo menos um deles deve ser categórico
- Pelo menos 1 target (não há necessidade de mais que 1 target e pode ser um target categórico 
ou numérico, a sua escolha)
- Pelo menos 150 exemplos
- Os dados não podem ser sintéticos


### Expectativas

<b>Expectativas:</b> É esperado que o trabalho siga as boas práticas apresentadas na disciplina, apresente 
resultados de forma gráfica, use seções bem nomeadas e bem divididas no notebook de Jupyter e 
encerre a entrega com uma seção chamada “Conclusões e principais aprendizados”. O trabalho deve 
ter uma seção chamada “Descrição do uso de IA neste trabalho” detalhando como foi a interação com 
ferramentas de IA para a construção dos textos e códigos.

### Pontuação base

<b>Pontuação base:</b> 7 (válida apenas caso o objetivo tenha sido realizado)
Descontos usuais da pontuação base:
- Alguma expectativa não foi atingida
- Trabalho não seguiu boas práticas apresentadas na disciplina. Lembre‑se de revisar o material 
apresentado e suas anotações durante a aula
- Trabalho não apresentou resultados de forma gráfica
- Não guiou o leitor na entrega
- Não interpretou e discutiu os resultados
- Gráfico sem legenda em um ou mais eixos
- Código que acusa erro ao rodar na ordem de cima para baixo
- Entrega sem referências
- Uso incorreto ou omisso de citações (incluindo falta de citação sobre uso de IA)
- Não soube explicar sobre estratégias e códigos utilizados na entrega
- Respostas incorretas às perguntas realizadas pelo professor durante o momento avaliativo
- Respostas incorretas às perguntas realizadas pelo Perguntador 2000™ durante o momento 
avaliativo
- Atraso na entrega do trabalho no Teams

### Pontuação complementar

<b>Pontuação complementar:</b> 3 (cada item abaixo pode valer até no máximo 1 ponto)
- Realizou uma boa análise exploratória dos dados e comparou o desempenho dos modelos com 
relação a um modelo baseline
- Checou como o desempenho dos modelos varia utilizando ou não utilizando dados normalizados de diferentes maneiras
- Checou o desempenho dos modelos variando o número de vizinhos
- Checou o desempenho dos modelos variando a função de distância (considerando pelo menos 
3 funções diferentes)
- Checou o desempenho dos modelos variando a estratégia de codificação do(s) atributo(s) categórico(s)



### Observações e Lembretes

<b>Motivação e aprendizado:</b> Esta é uma tarefa introdutória da disciplina, sendo assim, a motivação 
principal é justamente iniciar sua jornada no campo do Aprendizado de Máquina.

<b>Sugestão de uso de IA nesta tarefa:</b> Um bom uso de IA pode ser para encontrar conjuntos 
de dados de seu interesse. Fora isso, debater com a IA sobre quais atividades complementares fazer, 
ponderando aprendizado e interesse, também é uma boa estratégia.

<b>Lembretes:</b>
- “Guiar o leitor” não é sinônimo de “escrever um livro”. É perfeitamente possível guiar o leitor 
com textos curtos e diretos. Um texto que guia o leitor é aquele que deixa claro o que será 
realizado antes de realizar a tarefa e depois discute o que foi observado.
- Citações devem aparecer o mais próximo possível do local onde a informação foi apresentada 
ao leitor. Ter apenas uma seção de referências sem as devidas citações no texto não é a maneira 
correta de redigir um texto científico e implicará em descontos na pontuação.

## Introdução

### Importando bibliotecas

In [ ]:
import pandas as pd

### Lendo o arquivo

A primeira coisa que precisamos fazer é realizar a leitura do arquivo que vamos utilizar. Utilizamos o método `pd.read_csv()` para ler o arquivo `combined.csv`. É importante que o arquivo csv esteja na mesma pasta que esse notebook!!

In [ ]:
arquivo = "combined.csv"
df = pd.read_csv(arquivo, sep=",", header=None)
df

### Tratamento e filtro de dados

Agora que lemos o arquivo, podemos perceber que as colunas estão com números como seus nomes, e não categorias. Por outro lado, as categorias estão presentes na primeira linha do dataframe. Então, vamos renomear as colunas de acordo com a primeira linha, utilizando o método `.rename()`.\

In [ ]:
df = df.rename(columns={0: 'record_id', 1: 'month', 2: 'day', 3: 'year', 4: 'plot_id', 5: 'species_id', 6: 'sex', 
                   7: 'hindfoot_lenght', 8: 'weight', 9: 'genus', 10: 'species', 11: 'taxa', 12: 'plot_type'
                  })
df

Agora, vamos utilizar o método `.dropna()` para remover as linhas que tiverem informações faltando.

In [ ]:
df = df.dropna()
df

Como nós vamos trabalhar apenas com roedores, vamos filtrar o data frame para que sejam exibidas apenas as linhas cuja coluna `"taxa"` seja igual a `"Rodent"`, com o método `.loc[]`.

In [ ]:
logica = df['taxa'] == "Rodent"
df = df.loc[logica]
df

Antes de decidirmos quais serão as duas espécies que vamos prever, precisamos saber quantos dados de cada uma dessas espécies existem na tabela. Para isso, vamos utilizar o método `.values_counts()` na coluna `species`, para que seja possível visualizar quantos exemplares de cada espécie foram registrados.

In [ ]:
serie_species = pd.Series( df["species"] )
frequencia_species = serie_species.value_counts()#.sort_index()
print(frequencia_species)

Queremos que as espécies que vamos tentar prever tenham pelo menos 150 exemplares cada uma. Vamos filtrar o data frame novamente para que atenda esse requisito:

In [ ]:
# localizamos quais sao as especies que têm pelo menos de 150 exemplos
especies_desejadas = []
for especie, quantidade in frequencia_species.items():
    if quantidade >= 150:
        especies_desejadas.append( especie )

print(especies_desejadas)

Utilizamos o método `isin()` [2] para filtrar apenas as espécies que estejam dentro da lista de "espécies desejadas".

In [ ]:
logica = df['species'].isin( especies_desejadas )
df = df.loc[logica]
df

Para vizualizarmos novamente quais espécies ainda estão no data frame e quantos exemplares existem de cada uma, utilizamos o método `.value_counts()` novamente.

In [ ]:
serie_species = pd.Series( df["species"] )
frequencia_species = serie_species.value_counts()
print(frequencia_species)

Como ainda são muitas espécies e precisamos de apenas duas para fazer uma previsão de classificação binária, vamos escolher duas espécies. As espécies escolhidas foram a espécie `ordii`, com 2790 exemplares, e a espécie `spectabilis`, com 2023 exemplares. Vamos filtrar o data frame mais uma vez.

In [ ]:
df = df.loc[ df['species'].isin( ['ordii', 'spectabilis'] ) ]
df

## Uso de inteligência artificial

## Referências

[2] https://labex.io/pt/tutorials/pandas-pandas-filtering-data-596393